
# Week 3: Full-Corpus Financial Sentiment Analysis

**Owner:** Fatima Nasir

**Project:** Market Stress Early-Warning System for U.S. Equity Markets

**Focus:** NLP / Financial Sentiment Analysis

## Objective

This notebook builds on the Week 2 FinBERT prototype.

The goals are to:
1. Score all eligible SEC filing paragraphs using FinBERT.
2. Analyze Risk Factors and MD&A separately.
3. Measure sentiment changes between company filings.
4. Align filing dates with stock market trading days.
5. Save daily sentiment features for the team's market stress analysis.

We use the shared SEC dataset and do not download filings again.


In [2]:

# Step 1: Load shared SEC filing data

import pandas as pd
from pathlib import Path

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

data_path = ROOT / "data" / "processed" / "sec" / "paragraphs.parquet"

paras = pd.read_parquet(data_path)

print("Total paragraphs:", len(paras))
print("Companies:", paras["ticker"].unique())
print("Columns:", paras.columns.tolist())

display(paras.head(3))


Total paragraphs: 5600
Companies: ['AAPL' 'NVDA' 'MSFT' 'GOOGL' 'AMZN']
Columns: ['ticker', 'form', 'filing_date', 'accession', 'paragraph_id', 'section', 'text']


,ticker,form,filing_date,accession,paragraph_id,section,text
0,AAPL,10-Q,2022-01-28,0000320193-22-000007,1,front matter,Indicate by check mark whether the Registrant ...
1,AAPL,10-Q,2022-01-28,0000320193-22-000007,2,front matter,Indicate by check mark whether the Registrant ...
2,AAPL,10-Q,2022-01-28,0000320193-22-000007,3,front matter,Indicate by check mark whether the Registrant ...


## 2. Prepare Text for Sentiment Scoring

We use Risk Factors and MD&A paragraphs from the shared SEC dataset.

Following Week 2 peer-review feedback, paragraphs shorter than 15 words are excluded, and the two disclosure sections are analyzed separately.

In [3]:

# Step 2: Prepare the full text corpus

import re

sentiment_df = paras.dropna(subset=["text"]).copy()

sentiment_df["clean_text"] = sentiment_df["text"].apply(
    lambda text: re.sub(r"\s+", " ", str(text)).strip()
)

sentiment_df["word_count"] = (
    sentiment_df["clean_text"].str.split().str.len()
)

# Keep paragraphs with at least 15 words
sentiment_df = sentiment_df[
    sentiment_df["word_count"] >= 15
].copy()

# Identify Risk Factors and MD&A
risk_mask = sentiment_df["section"] == "Item 1A"

mdna_mask = (
    ((sentiment_df["form"] == "10-K") &
     (sentiment_df["section"] == "Item 7"))
    |
    ((sentiment_df["form"] == "10-Q") &
     (sentiment_df["section"] == "Item 2"))
)

sentiment_df = sentiment_df[
    risk_mask | mdna_mask
].copy()

sentiment_df["section_group"] = "MD&A"

sentiment_df.loc[
    sentiment_df["section"] == "Item 1A",
    "section_group"
] = "Risk Factors"

print("Total paragraphs ready:", len(sentiment_df))

display(
    pd.crosstab(
        sentiment_df["ticker"],
        sentiment_df["section_group"]
    )
)


Total paragraphs ready: 2802


section_group,MD&A,Risk Factors
ticker,,
AAPL,191,112
AMZN,280,381
GOOGL,475,165
MSFT,335,267
NVDA,310,286


## 3. Full-Corpus Sentiment Scoring

We use the pretrained FinBERT model to classify all eligible SEC filing paragraphs as positive, negative, or neutral.

Risk Factors and MD&A remain separate because standard risk disclosures may be classified as negative even when financial conditions have not changed.

In [5]:

# Step 3: Score the full SEC corpus with FinBERT

from transformers import pipeline

model_name = "ProsusAI/finbert"

sentiment_model = pipeline(
    "text-classification",
    model=model_name,
    tokenizer=model_name,
    device=-1
)

# Score all eligible paragraphs in batches
predictions = sentiment_model(
    sentiment_df["clean_text"].tolist(),
    truncation=True,
    max_length=512,
    batch_size=8
)

sentiment_df["sentiment"] = [
    p["label"] for p in predictions
]

sentiment_df["confidence"] = [
    round(p["score"], 4) for p in predictions
]

print("Total paragraphs scored:", len(sentiment_df))

display(
    pd.crosstab(
        sentiment_df["section_group"],
        sentiment_df["sentiment"]
    )
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Total paragraphs scored: 2802


sentiment,negative,neutral,positive
section_group,,,
MD&A,311,913,367
Risk Factors,829,344,38


In [6]:

# Step 4: Save full-corpus FinBERT predictions

output_dir = ROOT / "data" / "processed" / "features"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "sec_sentiment_scored.parquet"

sentiment_df.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Rows saved:", len(sentiment_df))
print("File exists:", output_path.exists())


Saved: /Users/broy/Documents/GitHub/RGBFS/data/processed/features/sec_sentiment_scored.parquet
Rows saved: 2802
File exists: True


## 5. Filing-Level Sentiment and Tone Changes

We calculate average sentiment for each company, filing date, and disclosure section.

Risk Factors and MD&A are analyzed separately to avoid mixing standard risk disclosures with management's discussion of financial performance.

Changes in sentiment between filings may provide useful signals for market stress analysis.


### Filtering Repeated Risk Factors

Risk Factors paragraphs are compared with earlier filings from the same company. Exact duplicates and paragraphs with 90% or higher text similarity are excluded to avoid repeated disclosures affecting sentiment results.

MD&A paragraphs remain unchanged. Filings without new Risk Factors are assigned missing sentiment values (NaN) rather than zero.

The 90% similarity threshold is a heuristic, so some meaningful wording changes may require further review.


In [7]:

# Step 4.5: Identify new or changed Risk Factors

from difflib import SequenceMatcher

risk_df = sentiment_df[
    sentiment_df["section_group"] == "Risk Factors"
].sort_values(
    ["ticker", "filing_date", "accession", "paragraph_id"]
).copy()

risk_df["is_new"] = True

# Compare with earlier filings from the same company
for ticker, group in risk_df.groupby("ticker"):
    previous_texts = []

    for filing_date, filing in group.groupby(
        "filing_date", sort=True
    ):
        current_texts = filing["clean_text"].fillna("").tolist()

        for idx, text in zip(filing.index, current_texts):
            repeated = any(
                SequenceMatcher(None, text, old).ratio() >= 0.90
                for old in previous_texts
            )
            risk_df.loc[idx, "is_new"] = not repeated

        # Add current filing after comparisons
        previous_texts.extend(current_texts)

sentiment_df["is_new"] = True
sentiment_df.loc[risk_df.index, "is_new"] = risk_df["is_new"]

print("Risk Factors retained:", risk_df["is_new"].sum())
print("Repeated or near-duplicate:",
      (~risk_df["is_new"]).sum())
print("MD&A retained:",
      sentiment_df[
          sentiment_df["section_group"] == "MD&A"
      ]["is_new"].sum())


Risk Factors retained: 647
Repeated or near-duplicate: 564
MD&A retained: 1591


In [8]:

# Step 5: Calculate filing-level sentiment

score_map = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}

sentiment_df["sentiment_score"] = sentiment_df["sentiment"].map(score_map)

# Keep MD&A and new Risk Factors
filtered_df = sentiment_df[sentiment_df["is_new"]].copy()

filing_keys = [
    "ticker", "accession", "filing_date", "form", "section_group"
]

# Calculate sentiment using new paragraphs
filing_summary = (
    filtered_df.groupby(filing_keys)
    .agg(
        paragraphs=("sentiment_score", "size"),
        average_sentiment=("sentiment_score", "mean")
    )
    .reset_index()
)

# Preserve filings with no new paragraphs
all_filings = sentiment_df[filing_keys].drop_duplicates()

filing_summary = all_filings.merge(
    filing_summary,
    on=filing_keys,
    how="left"
)

filing_summary = filing_summary.sort_values(
    ["ticker", "section_group", "form", "filing_date"]
)

# Compare only the same filing types
filing_summary["tone_change"] = (
    filing_summary.groupby(
        ["ticker", "section_group", "form"]
    )["average_sentiment"].diff()
)

print("Filing-level records:", len(filing_summary))
print("Missing sentiment:", filing_summary["average_sentiment"].isna().sum())
display(filing_summary.head(15))


Filing-level records: 40
Missing sentiment: 3


,ticker,accession,filing_date,form,section_group,paragraphs,average_sentiment,tone_change
7,AAPL,0000320193-22-000108,2022-10-28,10-K,MD&A,49.0,0.122449,NaN
0,AAPL,0000320193-22-000007,2022-01-28,10-Q,MD&A,46.0,0.108696,NaN
2,AAPL,0000320193-22-000059,2022-04-29,10-Q,MD&A,50.0,0.120000,0.011304
4,AAPL,0000320193-22-000070,2022-07-29,10-Q,MD&A,46.0,-0.021739,-0.141739
6,AAPL,0000320193-22-000108,2022-10-28,10-K,Risk Factors,109.0,-0.550459,NaN
1,AAPL,0000320193-22-000007,2022-01-28,10-Q,Risk Factors,1.0,0.000000,NaN
3,AAPL,0000320193-22-000059,2022-04-29,10-Q,Risk Factors,NaN,NaN,NaN
5,AAPL,0000320193-22-000070,2022-07-29,10-Q,Risk Factors,NaN,NaN,NaN
33,AMZN,0001018724-22-000005,2022-02-04,10-K,MD&A,78.0,0.012821,NaN
34,AMZN,0001018724-22-000013,2022-04-29,10-Q,MD&A,69.0,-0.144928,NaN


In [9]:

# Step 6: Check available market data files

processed_dir = ROOT / "data" / "processed"

for file in sorted(processed_dir.rglob("*")):
    if file.is_file() and file.suffix.lower() in [".csv", ".parquet"]:
        print(file.relative_to(ROOT))


data/processed/daily_panel.parquet
data/processed/features/connectedness_60d.parquet
data/processed/features/sec_sentiment_scored.parquet
data/processed/network_features.parquet
data/processed/returns_daily.parquet
data/processed/returns_market.parquet
data/processed/returns_quality.csv
data/processed/returns_summary.csv
data/processed/sec/filings.csv
data/processed/sec/paragraphs.parquet
data/processed/sec/problems.csv
data/processed/sp500_close.parquet
data/processed/sp500_coverage_report.csv
data/processed/sp500_membership.parquet
data/processed/sp500_volume.parquet


In [10]:

# Step 7: Inspect market trading dates

market = pd.read_parquet(
    ROOT / "data" / "processed" / "daily_panel.parquet"
)

print("Market data shape:", market.shape)
print("Columns:", market.columns.tolist())
print("Index:", market.index.name)

display(market.head(3))


Market data shape: (7005, 28)
Columns: ['SPY', 'XLB', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLU', 'XLV', 'XLY', 'SPY_ret', 'XLB_ret', 'XLE_ret', 'XLF_ret', 'XLI_ret', 'XLK_ret', 'XLP_ret', 'XLU_ret', 'XLV_ret', 'XLY_ret', 'STLFSI4', 'NFCI', 'VIXCLS', 'BAMLH0A0HYM2', 'T10Y2Y', 'stress', 'avg_corr_60d', 'absorption_250d']
Index: date


,SPY,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLV,XLY,...,XLV_ret,XLY_ret,STLFSI4,NFCI,VIXCLS,BAMLH0A0HYM2,T10Y2Y,stress,avg_corr_60d,absorption_250d
date,,,,,,,,,,,,,,,,,,,,,
1999-01-05,76.952873,6.040748,5.646688,11.248714,15.244631,12.562199,14.132487,5.807240,17.525572,9.684029,...,0.015541,0.016707,0.3871,-0.205,24.46,NaN,0.10,0,NaN,NaN
1999-01-06,78.808121,6.147099,5.833011,11.605341,15.572261,12.933373,14.303343,5.867513,17.816631,9.959072,...,0.016471,0.028006,0.3871,-0.205,23.34,NaN,0.11,0,NaN,NaN
1999-01-07,78.421585,6.100304,5.806393,11.783657,15.437355,12.892776,14.140618,5.846420,17.733465,9.947609,...,-0.004679,-0.001152,0.3871,-0.205,24.37,NaN,0.14,0,NaN,NaN


## 6. Align SEC Filings with Market Trading Days

SEC filings may be published on weekends or market holidays. We align each filing to the next available trading day in our market dataset.

This allows us to combine filing sentiment with daily market indicators.

In [11]:

# Step 8: Align SEC filings with trading days

import numpy as np

# Get trading dates from the market data
trading_days = pd.DatetimeIndex(
    pd.to_datetime(market.index)
).dropna().unique().sort_values()

# Convert filing dates to datetime
filing_summary["filing_date"] = pd.to_datetime(
    filing_summary["filing_date"]
)

# Find the next trading day on or after each filing date
positions = trading_days.searchsorted(
    filing_summary["filing_date"],
    side="left"
)

# Avoid dates outside the market data range
valid = positions < len(trading_days)

filing_summary["trading_date"] = pd.NaT

filing_summary.loc[valid, "trading_date"] = (
    trading_days.take(positions[valid]).to_numpy()
)

print("Total filing-section records:", len(filing_summary))
print("Successfully aligned:", filing_summary["trading_date"].notna().sum())

display(
    filing_summary[
        ["ticker", "filing_date", "trading_date",
         "section_group", "average_sentiment", "tone_change"]
    ].head(15)
)


Total filing-section records: 40
Successfully aligned: 40


,ticker,filing_date,trading_date,section_group,average_sentiment,tone_change
7,AAPL,2022-10-28,2022-10-28,MD&A,0.122449,NaN
0,AAPL,2022-01-28,2022-01-28,MD&A,0.108696,NaN
2,AAPL,2022-04-29,2022-04-29,MD&A,0.120000,0.011304
4,AAPL,2022-07-29,2022-07-29,MD&A,-0.021739,-0.141739
6,AAPL,2022-10-28,2022-10-28,Risk Factors,-0.550459,NaN
1,AAPL,2022-01-28,2022-01-28,Risk Factors,0.000000,NaN
3,AAPL,2022-04-29,2022-04-29,Risk Factors,NaN,NaN
5,AAPL,2022-07-29,2022-07-29,Risk Factors,NaN,NaN
33,AMZN,2022-02-04,2022-02-04,MD&A,0.012821,NaN
34,AMZN,2022-04-29,2022-04-29,MD&A,-0.144928,NaN


## 7. Create Daily Sentiment Features

We carry forward the latest available filing sentiment for each company and section across trading days.

MD&A and Risk Factors are kept separate. Days before a company's first filing remain missing because no sentiment information is available yet.

In [12]:

# Step 9: Build daily sentiment features

# Use trading days from 2022 onward
dates = pd.DataFrame({
    "date": trading_days[
        trading_days >= pd.Timestamp("2022-01-01")
    ]
})
dates["date"] = pd.to_datetime(dates["date"]).astype("datetime64[ns]")

filing_summary["trading_date"] = (
    pd.to_datetime(filing_summary["trading_date"])
    .astype("datetime64[ns]")
)

daily_tables = []

for (ticker, section), group in filing_summary.groupby(
    ["ticker", "section_group"]
):
    group = group.sort_values("trading_date").copy()

    # One row per trading day, using the latest filing
    daily = pd.merge_asof(
        dates,
        group[
            ["trading_date", "average_sentiment",
             "tone_change", "paragraphs"]
        ],
        left_on="date",
        right_on="trading_date",
        direction="backward"
    )

    daily["ticker"] = ticker
    daily["section_group"] = section

    daily_tables.append(daily)

daily_sentiment = pd.concat(
    daily_tables,
    ignore_index=True
)

daily_sentiment = daily_sentiment.rename(
    columns={"trading_date": "source_filing_trading_date"}
)

daily_sentiment = daily_sentiment[
    ["date", "ticker", "section_group",
     "average_sentiment", "tone_change",
     "paragraphs", "source_filing_trading_date"]
]

print("Daily sentiment rows:", len(daily_sentiment))
print("Companies:", daily_sentiment["ticker"].nunique())

display(daily_sentiment.dropna(subset=["average_sentiment"]).head(10))


Daily sentiment rows: 12180
Companies: 5


,date,ticker,section_group,average_sentiment,tone_change,paragraphs,source_filing_trading_date
18,2022-01-28,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
19,2022-01-31,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
20,2022-02-01,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
21,2022-02-02,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
22,2022-02-03,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
23,2022-02-04,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
24,2022-02-07,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
25,2022-02-08,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
26,2022-02-09,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
27,2022-02-10,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28


In [13]:

# Step 10: Save daily sentiment features

output_dir = ROOT / "data" / "processed" / "features"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "sec_sentiment_daily.csv"
parquet_path = output_dir / "sec_sentiment_daily.parquet"

daily_sentiment.to_csv(csv_path, index=False)
daily_sentiment.to_parquet(parquet_path, index=False)

print("CSV saved:", csv_path.exists())
print("Parquet saved:", parquet_path.exists())
print("Total rows:", len(daily_sentiment))
print("Companies:", daily_sentiment["ticker"].nunique())


CSV saved: True
Parquet saved: True
Total rows: 12180
Companies: 5


In [14]:

# Step 11: Finalize daily sentiment features

# Keep only 2022 trading days
daily_final = daily_sentiment[
    (daily_sentiment["date"] >= "2022-01-01") &
    (daily_sentiment["date"] < "2023-01-01")
].copy()

# Delay each filing's sentiment by one trading day
# to avoid using same-day information too early
daily_final = daily_final.sort_values(
    ["ticker", "section_group", "date"]
)

feature_cols = [
    "average_sentiment",
    "tone_change",
    "paragraphs",
    "source_filing_trading_date"
]

daily_final[feature_cols] = (
    daily_final.groupby(["ticker", "section_group"])[feature_cols]
    .shift(1)
)

# Overwrite preliminary files with finalized data
daily_final.to_csv(csv_path, index=False)
daily_final.to_parquet(parquet_path, index=False)

print("Final rows:", len(daily_final))
print("Companies:", daily_final["ticker"].nunique())
print("Date range:", daily_final["date"].min(),
      "to", daily_final["date"].max())
print("CSV saved:", csv_path.exists())
print("Parquet saved:", parquet_path.exists())


Final rows: 2510
Companies: 5
Date range: 2022-01-03 00:00:00 to 2022-12-30 00:00:00
CSV saved: True
Parquet saved: True


In [15]:

# Step 12: Verify final Week 3 output

check = pd.read_parquet(parquet_path)

print("Total rows:", len(check))
print("Duplicate date/company/section rows:",
      check.duplicated(["date", "ticker", "section_group"]).sum())

print("Sentiment values available:",
      check["average_sentiment"].notna().sum())

print("Companies:", sorted(check["ticker"].unique()))
print("Sections:", sorted(check["section_group"].unique()))

display(check.dropna(subset=["average_sentiment"]).head())


Total rows: 2510
Duplicate date/company/section rows: 0
Sentiment values available: 2052
Companies: ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'NVDA']
Sections: ['MD&A', 'Risk Factors']


,date,ticker,section_group,average_sentiment,tone_change,paragraphs,source_filing_trading_date
19,2022-01-31,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
20,2022-02-01,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
21,2022-02-02,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
22,2022-02-03,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
23,2022-02-04,AAPL,MD&A,0.108696,NaN,46.0,2022-01-28
